# Experimentos multivariados

Estudio Monte Carlo que compara modelos clasicos vectoriales contra **Chronos-2** (modo
conjunto, `ChronosMultivariateModel`) sobre 55 DGPs multivariados, con T en {50, 100, 200}
y R = 500 replicas. Metricas per-variable (sesgo, varianza, RMSE, MAE, CRPS, cobertura) y
conjuntas (Trace MSFE + avgCRPS marginal).

## Resumen

| Bloque | DGPs | T | R | Total series |
|--------|------|---|---|--------------|
| M-A — VAR(1) bivariado | 12 | 3 | 500 | 18,000 |
| M-B — VAR orden superior | 9 | 3 | 500 | 13,500 |
| M-C — Dimensionalidad creciente | 5 | 3 | 500 | 7,500 |
| M-D — VAR + GARCH diagonal | 9 | 3 | 500 | 13,500 |
| M-E — Cointegracion VECM | 8 | 3 | 500 | 12,000 |
| M-F — Eigenvalores complejos | 3 | 3 | 500 | 4,500 |
| M-G — Acople sistematico | 9 | 3 | 500 | 13,500 |
| **Total** | **55** | — | — | **82,500** |

**Bloques de experimentos:**
- **M-A** (12) — VAR(1) bivariado estacionario, distintos patrones de dependencia y Sigma.
- **M-B** (9) — VAR de orden superior y cerca de raiz unitaria.
- **M-C** (5) — Dimensionalidad creciente (k = 3..6).
- **M-D** (9) — VAR + GARCH diagonal (heteroscedasticidad condicional).
- **M-E** (8) — Cointegracion VECM bivariado, rango 1.
- **M-F** (3) — VAR con eigenvalores complejos (ciclos endogenos).
- **M-G** (9) — Patrones de acople: block-diagonal, hub-spoke, cadena triangular, dim x lag.

> Modelo clasico correcto por experimento (VAR/VECM/VAR+GARCH-diag) vs Chronos-2 conjunto.
> `VARDGP` asume media cero (sin constante/drift). Resultados cacheados en
> `results/multivariados/`: si el CSV existe, se carga sin re-simular.

In [ ]:
import os
import warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

import copy
import logging
import sys
import time
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch
from IPython.display import display

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tsa.vector_ar.var_model import VAR as SMVAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen

from mectesis.dgp import VARDGP, VARGARCHDiagonalDGP, VECMBivariateDGP
from mectesis.models import (
    VARModel, VECMModel, VARGARCHDiagonalModel,
    ChronosModel, ChronosMultivariateModel,
)
from mectesis.simulation import MultivariateMonteCarloEngine
from mectesis.metrics import trace_msfe, avg_marginal_crps  # noqa: F401 — usadas via engine

# Parametros globales
SEED    = 3649
H_BY_T  = {50: 6, 100: 18, 200: 24}
H_MAX   = 24
R_LIST  = [500]
T_LIST  = [50, 100, 200]
RESULTS = Path("results/multivariados")
RESULTS.mkdir(parents=True, exist_ok=True)

# Logging dual: notebook + archivo .log
log_path = RESULTS / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger().info
log(f"Log en: {log_path}")

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)

device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Cargando Chronos-2 en {device} (puede tardar ~30 s la primera vez)...")
_chronos_base = ChronosModel(device=device)
chronos_mv    = ChronosMultivariateModel(_chronos_base)
log("Chronos-2 listo.")

In [ ]:
# === Estilo de figuras: fan-plot consistente con el anexo de la tesis ===
from mectesis.empirical.describe import plot_forecast_fan
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "STIX Two Text", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "mathtext.rm": "serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 8,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CHRONOS_COLOR = "#9672B6"   # modelo fundacional (Chronos-2), en violeta
CLASSIC_COLOR = "#4C72B0"   # modelos clasicos, en azul


def _colors_for(forecasts):
    return {name: (CHRONOS_COLOR if "Chronos" in name else CLASSIC_COLOR)
            for name in forecasts}


def _fan_series(arr, name="y"):
    arr = np.asarray(arr, dtype=float)
    return pd.Series(arr, index=pd.RangeIndex(len(arr)), name=name)


def _fan_frame(arr, prefix="Y"):
    arr = np.asarray(arr, dtype=float)
    return pd.DataFrame(arr, index=pd.RangeIndex(arr.shape[0]),
                        columns=[f"{prefix}{j+1}" for j in range(arr.shape[1])])


def _arr(x):
    return None if x is None else np.asarray(x, dtype=float)


def _band80(model, horizon, **kw):
    if not getattr(model, "supports_intervals", False):
        return None, None
    try:
        return model.forecast_intervals(horizon, level=0.80, **kw)
    except NotImplementedError:
        return None, None


def _fan_fit_uni(model, y_train, horizon):
    model.fit(np.asarray(y_train, dtype=float))
    mean = np.asarray(model.forecast(horizon), dtype=float)
    lo, hi = _band80(model, horizon)
    return {"mean": mean, "lo80": _arr(lo), "hi80": _arr(hi)}


def _fan_fit_multi(model, Y_train, horizon):
    model.fit(np.asarray(Y_train, dtype=float))
    mean = np.asarray(model.forecast(horizon), dtype=float)
    lo, hi = _band80(model, horizon)
    return {"mean": mean, "lo80": _arr(lo), "hi80": _arr(hi)}


def _fan_fit_cov(model, y_train, X_train, X_future, horizon):
    has_cov = getattr(model, "supports_covariates", False)
    model.fit(np.asarray(y_train, dtype=float),
              **({"X_train": X_train} if has_cov else {}))
    pkw = {"X_future": X_future} if has_cov else {}
    mean = np.asarray(model.forecast(horizon, **pkw), dtype=float)
    lo, hi = _band80(model, horizon, **pkw)
    return {"mean": mean, "lo80": _arr(lo), "hi80": _arr(hi)}


def _fan_legend(fig, with_cov=False):
    band_c = Patch(facecolor=CHRONOS_COLOR, alpha=0.30, edgecolor="none")
    band_k = Patch(facecolor=CLASSIC_COLOR, alpha=0.30, edgecolor="none")
    handles = [
        Line2D([0], [0], color="#1f4068", lw=1.6),
        Line2D([0], [0], color="#1f4068", lw=1.6, ls="--", alpha=0.6),
        Line2D([0], [0], color=CHRONOS_COLOR, lw=2.2),
        Line2D([0], [0], color=CLASSIC_COLOR, lw=2.2),
        (band_c, band_k),
    ]
    labels = ["Observado", "Realizado", "Chronos-2", "Modelo clasico", "Banda 80%"]
    if with_cov:
        handles.append(Line2D([0], [0], color="#3a3a3a", lw=1.2))
        labels.append("Covariable(s)")
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
               fontsize=7, frameon=True, framealpha=0.9, bbox_to_anchor=(0.5, 1.0),
               columnspacing=1.1, handletextpad=0.4,
               handler_map={tuple: HandlerTuple(ndivide=None)})

In [ ]:
# ─── Funciones auxiliares multivariadas ─────────────────────────────────────

def _cache_path(exp_id: str, T: int, R: int) -> Path:
    return RESULTS / f"exp_{exp_id.replace('.', '_')}_T{T}_R{R}.csv"


def _save_results_mv(results: dict, path: Path):
    """Guarda {model: {var_idx: DataFrame}} como CSV con columnas 'model','var'."""
    frames = []
    for mname, var_dict in results.items():
        for var_idx, df in var_dict.items():
            tmp = df.copy()
            tmp.insert(0, "var", var_idx)
            tmp.insert(0, "model", mname)
            frames.append(tmp)
    pd.concat(frames, ignore_index=True).to_csv(path, index=False)


def _load_results_mv(path: Path) -> dict:
    df = pd.read_csv(path)
    results = {}
    for mname, mgrp in df.groupby("model", sort=False):
        results[mname] = {}
        for var_idx, vgrp in mgrp.groupby("var", sort=True):
            results[mname][int(var_idx)] = (
                vgrp.drop(columns=["model", "var"]).reset_index(drop=True)
            )
    return results


def run_exp_mv(dgp, make_models_fn, dgp_params, exp_id,
               T_list=None, R_list=None, H_by_T=None, seed=SEED):
    """
    Corre MC multivariado para todas las combinaciones (T, R).
    T_list/R_list/H_by_T per-experimento sobrescriben los globales.
    """
    T_list = T_list if T_list is not None else T_LIST
    R_list = R_list if R_list is not None else R_LIST
    H_by_T = H_by_T if H_by_T is not None else H_BY_T

    n_runs = len(T_list) * len(R_list)
    combos = ", ".join(
        f"(T={t}, H={H_by_T.get(t, H_MAX)}, R={r})"
        for t in T_list for r in R_list
    )
    log(f"Exp {exp_id}: {n_runs} ejecucion(es) -> {combos}")

    all_results = {}
    for T in T_list:
        h = H_by_T.get(T, H_MAX)
        for R in R_list:
            cache = _cache_path(exp_id, T, R)
            if cache.exists():
                log(f"  T={T} H={h}, R={R}: cargando {cache.name}")
                all_results[(T, R)] = _load_results_mv(cache)
                continue
            log(f"  T={T} H={h}, R={R}: simulando...")
            dgp.rng = np.random.default_rng(seed)
            models = make_models_fn(T)
            engine = MultivariateMonteCarloEngine(dgp, models, seed=seed)
            t0 = time.time()
            results = engine.run_monte_carlo(R, T, h, dgp_params, verbose=False)
            log(f"  T={T} H={h}, R={R}: OK ({time.time()-t0:.0f}s)")
            _save_results_mv(results, cache)
            all_results[(T, R)] = results
    return all_results


# ─── Bloques v3 (Corto / Medio / Largo) ─────────────────────────────────────

BLOCK_DEFS = [("C", 1, 6), ("M", 7, 18), ("L", 19, 24)]
METRICS_V3  = ["bias", "variance", "rmse", "crps"]


def compute_blocks_mv(results_TR: dict) -> dict:
    """Promedios por bloque h por variable: {model: {var: {blk: Series}}}"""
    out = {}
    for mname, var_dict in results_TR.items():
        out[mname] = {}
        for var_idx, df in var_dict.items():
            df_h = df[df["horizon"] != "avg_all"].copy()
            df_h["horizon"] = pd.to_numeric(df_h["horizon"], errors="coerce")
            blks = {}
            for blk, h1, h2 in BLOCK_DEFS:
                mask = (df_h["horizon"] >= h1) & (df_h["horizon"] <= h2)
                blks[blk] = df_h[mask].mean(numeric_only=True)
            out[mname][var_idx] = blks
    return out


def build_grid_table_mv(all_results, classical_name: str, chronos_name: str = "Chronos-2 (joint)",
                         var_names=None):
    """
    Tabla 1 (per-variable) por (T, Variable, Modelo) con metricas por bloque h
    y marcador C/T. Ignora la fila virtual var=-1 (metricas joint, ver tabla 2).
    """
    rows = []
    for (T, R), res_TR in sorted(all_results.items()):
        blk_data = compute_blocks_mv(res_TR)
        var_idx_set = set()
        for var_dict in blk_data.values():
            var_idx_set.update(var_dict.keys())
        var_idx_set = {v for v in var_idx_set if v >= 0}   # excluir joint

        for var_idx in sorted(var_idx_set):
            vname = var_names[var_idx] if var_names else f"Y{var_idx+1}"
            cl_blks = blk_data.get(classical_name, {}).get(var_idx, {})
            ch_blks = blk_data.get(chronos_name, {}).get(var_idx, {})

            for mname, var_blks in blk_data.items():
                if var_idx not in var_blks:
                    continue
                blks = var_blks[var_idx]
                row = {"T": T, "Variable": vname, "Modelo": mname}
                for blk, h1, h2 in BLOCK_DEFS:
                    s = blks.get(blk, pd.Series(dtype=float))
                    for m in METRICS_V3:
                        row[f"{m}_{blk}"] = (
                            round(float(s[m]), 4)
                            if m in s.index and pd.notna(s[m]) else np.nan
                        )
                    cl_s = cl_blks.get(blk, pd.Series(dtype=float))
                    ch_s = ch_blks.get(blk, pd.Series(dtype=float))
                    for m in ["rmse", "crps"]:
                        cv = float(cl_s[m]) if m in cl_s.index and pd.notna(cl_s[m]) else np.nan
                        hv = float(ch_s[m]) if m in ch_s.index and pd.notna(ch_s[m]) else np.nan
                        if np.isnan(cv) or np.isnan(hv):
                            row[f"best_{m}_{blk}"] = np.nan
                        else:
                            row[f"best_{m}_{blk}"] = "C" if cv <= hv else "T"
                rows.append(row)

    df_out = pd.DataFrame(rows).set_index(["T", "Variable", "Modelo"])
    display(df_out.style.format(precision=4, na_rep="—"))


def build_grid_table_mv_joint(all_results, classical_name: str,
                               chronos_name: str = "Chronos-2 (joint)"):
    """
    Tabla 2 (multivariada conjunta) por (T, Modelo) con Trace MSFE y avgCRPS
    promediados por bloque h (C / M / L), mas marcadores C/T por bloque
    indicando que modelo gana (mismo formato que build_grid_table_mv).
    Lee la fila virtual var=-1 que el engine inyecta.
    """
    rows = []
    for (T, R), res_TR in sorted(all_results.items()):
        blk_data = compute_blocks_mv(res_TR)
        cl_blks = blk_data.get(classical_name, {}).get(-1, {})
        ch_blks = blk_data.get(chronos_name, {}).get(-1, {})

        for mname, var_blks in blk_data.items():
            joint = var_blks.get(-1, None)
            if joint is None:
                continue
            row = {"T": T, "Modelo": mname}
            for blk, h1, h2 in BLOCK_DEFS:
                s = joint.get(blk, pd.Series(dtype=float))
                for m in ["trace_msfe", "avg_crps"]:
                    row[f"{m}_{blk}"] = (
                        round(float(s[m]), 4)
                        if m in s.index and pd.notna(s[m]) else np.nan
                    )
                # Marcador C/T: gana clasico (C) si su metrica es <=, sino Chronos (T)
                cl_s = cl_blks.get(blk, pd.Series(dtype=float))
                ch_s = ch_blks.get(blk, pd.Series(dtype=float))
                for m in ["trace_msfe", "avg_crps"]:
                    cv = float(cl_s[m]) if m in cl_s.index and pd.notna(cl_s[m]) else np.nan
                    hv = float(ch_s[m]) if m in ch_s.index and pd.notna(ch_s[m]) else np.nan
                    if np.isnan(cv) or np.isnan(hv):
                        row[f"best_{m}_{blk}"] = np.nan
                    else:
                        row[f"best_{m}_{blk}"] = "C" if cv <= hv else "T"
            rows.append(row)

    if not rows:
        log("  [tabla joint] sin filas — el CSV no contiene metricas joint "
            "(probablemente generado por una version anterior del engine; "
            "borrar y re-ejecutar para incluirlas).")
        return

    df_out = pd.DataFrame(rows).set_index(["T", "Modelo"])
    display(df_out.style.format(precision=4, na_rep="—"))


def plot_simulation_mv(dgp, models, dgp_params, var_names=None,
                        title="", T_vis=100, seed=SEED):
    H_vis = H_BY_T.get(T_vis, H_MAX)
    dgp_r = copy.deepcopy(dgp)
    dgp_r.rng = np.random.default_rng(seed + 99991)
    Y = np.asarray(dgp_r.simulate(T=T_vis, **dgp_params), dtype=float)
    k = Y.shape[1]
    split = T_vis - H_vis
    Yf = _fan_frame(Y)
    forecasts = {}
    for m in models:
        try:
            forecasts[m.name] = _fan_fit_multi(m, Y[:split], H_vis)
        except Exception as e:
            log(f"  [plot] {m.name} fallo: {e}")
    fig, axes = plt.subplots(k, 1, figsize=(10, 2.8 * k), squeeze=False, sharex=True)
    for j in range(k):
        ax = axes[j, 0]
        vname = var_names[j] if var_names else f"Y{j+1}"
        plot_forecast_fan(Yf.iloc[:, j], forecasts, origin_idx=split, horizon=H_vis,
                          history_tail=split, title=(title if j == 0 else ""),
                          ax=ax, variable_idx=j, colors=_colors_for(forecasts),
                          show_legend=False)
        ax.set_ylabel(vname, fontsize=8)
    _fan_legend(fig)
    plt.tight_layout()
    plt.show()

# ─── Verificacion DGP multivariada ──────────────────────────────────────────

def verify_dgp_mv(label, dgp, dgp_params, checks):
    log(f"{'-'*60}")
    log(f"VERIFICACION DGP: {label}")
    log(f"{'-'*60}")
    dgp_copy = copy.deepcopy(dgp)
    dgp_copy.rng = np.random.default_rng(7777)
    try:
        y_long = dgp_copy.simulate(T=500, **dgp_params)
    except Exception as e:
        log(f"  [FAIL] simulate() lanzo excepcion: {e}")
        return
    n_fail = 0
    for check_name, check_fn in checks:
        try:
            ok, msg = check_fn(y_long, dgp, dgp_params)
        except Exception as e:
            ok, msg = False, f"excepcion inesperada: {e}"
        tag = "PASS" if ok else "FAIL"
        log(f"  [{tag}] {check_name}: {msg}")
        if not ok:
            n_fail += 1
    if n_fail == 0:
        log("  -> TODAS LAS VERIFICACIONES PASARON")
    else:
        log(f"  -> {n_fail} FALLO(S)")


# Helpers individuales
def _companion_eigvals(A_list):
    A_list = [np.asarray(A, float) for A in A_list]
    k = A_list[0].shape[0]
    p = len(A_list)
    C = np.zeros((k * p, k * p))
    C[:k, :] = np.hstack(A_list)
    if p > 1:
        C[k:, :k * (p - 1)] = np.eye(k * (p - 1))
    return np.linalg.eigvals(C)


def chk_var_stability(y, dgp, dgp_params):
    A_list = getattr(dgp, "A_list", None)
    if A_list is None and hasattr(dgp, "A1"):
        A_list = [dgp.A1]
    if A_list is None:
        return True, "no aplica"
    eig = _companion_eigvals(A_list)
    mod_max = float(np.max(np.abs(eig)))
    ok = mod_max < 0.999
    return ok, f"max|lambda companion|={mod_max:.4f} (umbral 0.999)"


def chk_var_near_unit_root(y, dgp, dgp_params):
    """Para M-B.5/6: aceptar 0.95-0.999 sin marcar fail."""
    A_list = getattr(dgp, "A_list", None)
    if A_list is None and hasattr(dgp, "A1"):
        A_list = [dgp.A1]
    if A_list is None:
        return True, "no aplica"
    eig = _companion_eigvals(A_list)
    mod_max = float(np.max(np.abs(eig)))
    ok = mod_max < 0.9999
    return ok, f"max|lambda|={mod_max:.4f} (cerca unit root esperado)"


def chk_sigma_psd(y, dgp, dgp_params):
    Sigma = getattr(dgp, "Sigma", None)
    if Sigma is None:
        return True, "no aplica"
    Sigma = np.asarray(Sigma, float)
    try:
        np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError as e:
        return False, f"Sigma no es PSD: {e}"
    eigs = np.linalg.eigvalsh(Sigma)
    return float(eigs.min()) > 1e-8, f"min eig(Sigma)={float(eigs.min()):.6f}"


def chk_empirical_finite(y, dgp, dgp_params):
    if not np.all(np.isfinite(y)):
        return False, "y contiene NaN/inf"
    sd = np.std(y, axis=0)
    return bool(np.all(sd < 1e6)), f"std(y)={np.round(sd, 3).tolist()}"


def chk_garch_stationarity(y, dgp, dgp_params):
    alphas = getattr(dgp, "alphas", None)
    betas  = getattr(dgp, "betas", None)
    if alphas is None or betas is None:
        return True, "no aplica"
    s = np.asarray(alphas) + np.asarray(betas)
    ok = bool(np.all(s < 1.0))
    return ok, f"alpha_i+beta_i={np.round(s, 4).tolist()} (todos < 1)"


def chk_arch_lm_mv(y, dgp, dgp_params, nlags=5):
    """ARCH-LM por variable sobre residuos VAR(1)."""
    try:
        res = SMVAR(y).fit(maxlags=1, trend="c")
        resid = res.resid
    except Exception as e:
        return False, f"VAR(1) fit fallo: {e}"
    pvals = []
    for i in range(resid.shape[1]):
        try:
            _, pval, _, _ = het_arch(resid[:, i], nlags=nlags)
            pvals.append(float(pval))
        except Exception:
            pvals.append(np.nan)
    pv_arr = np.array([p for p in pvals if not np.isnan(p)])
    if len(pv_arr) == 0:
        return False, "todos los het_arch fallaron"
    ok = bool(np.all(pv_arr < 0.05))
    return ok, f"p-values ARCH-LM por eq.={np.round(pvals, 4).tolist()} (se espera <0.05)"


def chk_johansen_rank(y, dgp, dgp_params):
    expected_rank = getattr(dgp, "get_theoretical_properties", lambda: {})().get("coint_rank", 1)
    try:
        res = coint_johansen(y, det_order=0, k_ar_diff=1)
    except Exception as e:
        return False, f"Johansen fallo: {e}"
    trace_stats = res.lr1
    cv_95 = res.cvt[:, 1]  # critical values 95%
    # Encontramos el rango estimado: el primer r tal que trace_stat <= cv
    rank_est = 0
    for r in range(len(trace_stats)):
        if trace_stats[r] > cv_95[r]:
            rank_est = r + 1
    ok = rank_est == expected_rank
    return ok, f"rango estimado={rank_est}, esperado={expected_rank}, trace={np.round(trace_stats, 2).tolist()}"


def chk_individual_I1_vecm(y, dgp, dgp_params):
    msgs, ok_all = [], True
    for i in range(y.shape[1]):
        pv_lvl = float(adfuller(y[:, i], autolag="AIC")[1])
        pv_dif = float(adfuller(np.diff(y[:, i]), autolag="AIC")[1])
        ok_i = (pv_lvl > 0.05) and (pv_dif < 0.05)
        ok_all = ok_all and ok_i
        msgs.append(f"Y{i+1}: ADF_lvl={pv_lvl:.3f}, ADF_dif={pv_dif:.3f}")
    return ok_all, " | ".join(msgs)


def chk_coint_combination_I0(y, dgp, dgp_params):
    beta = getattr(dgp, "beta", None)
    if beta is None:
        return True, "no aplica"
    beta = np.asarray(beta, float)
    z = y @ beta
    pv = float(adfuller(z, autolag="AIC")[1])
    return pv < 0.05, f"ADF(beta'Y) p={pv:.4f} (se espera <0.05)"


def chk_fit_classical_mv(y, dgp, dgp_params, model_factory=None):
    """Verifica que el modelo clasico ajusta y forecastea sin NaN."""
    if model_factory is None:
        return True, "no aplica"
    try:
        m = model_factory()
        m.fit(y[:300])
        fc = m.forecast(horizon=6)
        ok = fc.shape == (6, y.shape[1]) and np.all(np.isfinite(fc))
        return ok, f"fit+forecast OK, shape={fc.shape}"
    except Exception as e:
        return False, str(e)


# ── Grupos de checks por familia ────────────────────────────────────────────

CHECKS_VAR = [
    ("Estabilidad VAR (eigenvalores companion)", chk_var_stability),
    ("Sigma PSD",                                chk_sigma_psd),
    ("Salida finita y std razonable",            chk_empirical_finite),
]
CHECKS_VAR_NEAR_UNIT_ROOT = [
    ("Cerca de raiz unitaria (warning OK)",      chk_var_near_unit_root),
    ("Sigma PSD",                                chk_sigma_psd),
    ("Salida finita",                            chk_empirical_finite),
]
CHECKS_VAR_GARCH = [
    ("Estabilidad VAR media",                    chk_var_stability),
    ("Estacionariedad GARCH (alpha+beta<1)",     chk_garch_stationarity),
    ("Efectos ARCH detectables (LM por eq.)",    chk_arch_lm_mv),
    ("Salida finita",                            chk_empirical_finite),
]
CHECKS_VECM = [
    ("Sigma PSD",                                chk_sigma_psd),
    ("Cada serie I(1)",                          chk_individual_I1_vecm),
    ("Combinacion beta'Y es I(0)",               chk_coint_combination_I0),
    ("Rango cointegracion (Johansen)",           chk_johansen_rank),
]

## Bloque M-A — VAR(1) bivariado estacionario (12 experimentos)

$$\mathbf{y}_t=A_1\,\mathbf{y}_{t-1}+\boldsymbol{\varepsilon}_t,\qquad \boldsymbol{\varepsilon}_t\sim\mathcal{N}(\mathbf{0},\Sigma)$$

Se varian A_1 (densidad y signo) y Sigma (correlacion contemporanea). Clasico: VAR(1).

**Experimentos:**
- **M-A.1** — VAR(1) baja interdependencia · clasico: VAR(1)
- **M-A.2** — VAR(1) alta interdependencia · clasico: VAR(1)
- **M-A.3** — Persistencias asimetricas sin acople · clasico: VAR(1)
- **M-A.4** — Causalidad unidireccional Y1->Y2 · clasico: VAR(1)
- **M-A.5** — Off-diagonal y Sigma negativos · clasico: VAR(1)
- **M-A.6** — Persistencia baja (cercano a ruido blanco) · clasico: VAR(1)
- **M-A.7** — Sigma contemporanea fuerte · clasico: VAR(1)
- **M-A.8** — Sigma=I (sin acople contemporaneo) · clasico: VAR(1)
- **M-A.9** — Sigma casi-colineal · clasico: VAR(1)
- **M-A.10** — Sigma fuertemente negativa · clasico: VAR(1)
- **M-A.11** — Causalidad Y2->Y1 (triangular inferior) · clasico: VAR(1)
- **M-A.12** — Anti-persistencia (eig real negativo) · clasico: VAR(1)

In [ ]:
try:
    # M-A.1 -- VAR(1) baja interdependencia
    A_list = [[[0.5,0.1],[0.1,0.5]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.1 -- VAR(1) baja interdependencia", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.1")
    log("\n" + "="*60 + "\nM-A.1 -- VAR(1) baja interdependencia\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.1 -- VAR(1) baja interdependencia")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.2 -- VAR(1) alta interdependencia
    A_list = [[[0.4,0.4],[0.4,0.4]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.2 -- VAR(1) alta interdependencia", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.2")
    log("\n" + "="*60 + "\nM-A.2 -- VAR(1) alta interdependencia\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.2 -- VAR(1) alta interdependencia")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.3 -- Persistencias asimetricas sin acople
    A_list = [[[0.7,0.0],[0.0,0.3]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.3 -- Persistencias asimetricas sin acople", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.3")
    log("\n" + "="*60 + "\nM-A.3 -- Persistencias asimetricas sin acople\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.3 -- Persistencias asimetricas sin acople")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.4 -- Causalidad unidireccional Y1->Y2
    A_list = [[[0.5,0.3],[0.0,0.4]]]
    Sigma  = [[1.0,0.2],[0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.4 -- Causalidad unidireccional Y1->Y2", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.4")
    log("\n" + "="*60 + "\nM-A.4 -- Causalidad unidireccional Y1->Y2\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.4 -- Causalidad unidireccional Y1->Y2")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.5 -- Off-diagonal y Sigma negativos
    A_list = [[[0.6,-0.3],[-0.3,0.6]]]
    Sigma  = [[1.0,-0.4],[-0.4,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.5 -- Off-diagonal y Sigma negativos", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.5")
    log("\n" + "="*60 + "\nM-A.5 -- Off-diagonal y Sigma negativos\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.5 -- Off-diagonal y Sigma negativos")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.6 -- Persistencia baja (cercano a ruido blanco)
    A_list = [[[0.2,0.1],[0.1,0.2]]]
    Sigma  = [[1.0,0.1],[0.1,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.6 -- Persistencia baja (cercano a ruido blanco)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.6")
    log("\n" + "="*60 + "\nM-A.6 -- Persistencia baja (cercano a ruido blanco)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.6 -- Persistencia baja (cercano a ruido blanco)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.7 -- Sigma contemporanea fuerte
    A_list = [[[0.5,0.1],[0.1,0.5]]]
    Sigma  = [[1.0,0.7],[0.7,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.7 -- Sigma contemporanea fuerte", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.7")
    log("\n" + "="*60 + "\nM-A.7 -- Sigma contemporanea fuerte\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.7 -- Sigma contemporanea fuerte")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.8 -- Sigma=I (sin acople contemporaneo)
    A_list = [[[0.5,0.1],[0.1,0.5]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.8 -- Sigma=I (sin acople contemporaneo)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.8")
    log("\n" + "="*60 + "\nM-A.8 -- Sigma=I (sin acople contemporaneo)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.8 -- Sigma=I (sin acople contemporaneo)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.9 -- Sigma casi-colineal
    A_list = [[[0.5,0.1],[0.1,0.5]]]
    Sigma  = [[1.0,0.9],[0.9,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.9 -- Sigma casi-colineal", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.9")
    log("\n" + "="*60 + "\nM-A.9 -- Sigma casi-colineal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.9 -- Sigma casi-colineal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.10 -- Sigma fuertemente negativa
    A_list = [[[0.5,0.1],[0.1,0.5]]]
    Sigma  = [[1.0,-0.7],[-0.7,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.10 -- Sigma fuertemente negativa", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.10")
    log("\n" + "="*60 + "\nM-A.10 -- Sigma fuertemente negativa\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.10 -- Sigma fuertemente negativa")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.10 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.11 -- Causalidad Y2->Y1 (triangular inferior)
    A_list = [[[0.5,0.0],[0.3,0.4]]]
    Sigma  = [[1.0,0.2],[0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.11 -- Causalidad Y2->Y1 (triangular inferior)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.11")
    log("\n" + "="*60 + "\nM-A.11 -- Causalidad Y2->Y1 (triangular inferior)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.11 -- Causalidad Y2->Y1 (triangular inferior)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.11 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-A.12 -- Anti-persistencia (eig real negativo)
    A_list = [[[-0.6,0.1],[0.1,-0.6]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-A.12 -- Anti-persistencia (eig real negativo)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-A.12")
    log("\n" + "="*60 + "\nM-A.12 -- Anti-persistencia (eig real negativo)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-A.12 -- Anti-persistencia (eig real negativo)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-A.12 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-B — VAR de orden superior y cerca de raiz unitaria (9 experimentos)

$$\mathbf{y}_t=\sum_{i=1}^{p}A_i\,\mathbf{y}_{t-i}+\boldsymbol{\varepsilon}_t,\qquad p\ge 2$$

Memoria larga y eigenvalores cerca del circulo unitario. Clasico: VAR(p).

**Experimentos:**
- **M-B.1** — VAR(2) baseline · clasico: VAR(2)
- **M-B.2** — VAR(2) cruzadas en lag 2 · clasico: VAR(2)
- **M-B.3** — VAR(3) decaimiento geometrico · clasico: VAR(3)
- **M-B.4** — VAR(4) memoria larga · clasico: VAR(4)
- **M-B.5** — VAR(1) cerca unit root · clasico: VAR(1)
- **M-B.6** — VAR(2) cerca unit root · clasico: VAR(2)
- **M-B.7** — VAR(2) k=3 con acople cruzado en lag 2 · clasico: VAR(2)
- **M-B.8** — VAR(5) k=2 memoria muy larga · clasico: VAR(5)
- **M-B.9** — VAR(1) k=2 casi-explosivo (max eig 0.99) · clasico: VAR(1)

In [ ]:
try:
    # M-B.1 -- VAR(2) baseline
    A_list = [[[0.5,0.2],[0.1,0.4]], [[0.1,0.0],[0.0,0.1]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=2: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.1 -- VAR(2) baseline", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.1")
    log("\n" + "="*60 + "\nM-B.1 -- VAR(2) baseline\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(2)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(2)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.1 -- VAR(2) baseline")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.2 -- VAR(2) cruzadas en lag 2
    A_list = [[[0.4,0.0],[0.0,0.4]], [[0.2,0.1],[0.1,0.2]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=2: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.2 -- VAR(2) cruzadas en lag 2", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.2")
    log("\n" + "="*60 + "\nM-B.2 -- VAR(2) cruzadas en lag 2\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(2)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(2)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.2 -- VAR(2) cruzadas en lag 2")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.3 -- VAR(3) decaimiento geometrico
    A_list = [[[0.3,0.0],[0.0,0.3]], [[0.2,0.0],[0.0,0.2]], [[0.1,0.0],[0.0,0.1]]]
    Sigma  = [[1.0,0.2],[0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=3: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.3 -- VAR(3) decaimiento geometrico", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.3")
    log("\n" + "="*60 + "\nM-B.3 -- VAR(3) decaimiento geometrico\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(3)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(3)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.3 -- VAR(3) decaimiento geometrico")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.4 -- VAR(4) memoria larga
    A_list = [[[0.25,0.0],[0.0,0.25]], [[0.2,0.0],[0.0,0.2]], [[0.15,0.0],[0.0,0.15]], [[0.1,0.0],[0.0,0.1]]]
    Sigma  = [[1.0,0.2],[0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=4: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.4 -- VAR(4) memoria larga", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.4")
    log("\n" + "="*60 + "\nM-B.4 -- VAR(4) memoria larga\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(4)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(4)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.4 -- VAR(4) memoria larga")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.5 -- VAR(1) cerca unit root
    A_list = [[[0.95,0.02],[0.02,0.93]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.5 -- VAR(1) cerca unit root", dgp, {}, CHECKS_VAR_NEAR_UNIT_ROOT)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.5")
    log("\n" + "="*60 + "\nM-B.5 -- VAR(1) cerca unit root\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.5 -- VAR(1) cerca unit root")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.6 -- VAR(2) cerca unit root
    A_list = [[[0.6,0.1],[0.1,0.6]], [[0.35,0.0],[0.0,0.33]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=2: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.6 -- VAR(2) cerca unit root", dgp, {}, CHECKS_VAR_NEAR_UNIT_ROOT)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.6")
    log("\n" + "="*60 + "\nM-B.6 -- VAR(2) cerca unit root\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(2)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(2)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.6 -- VAR(2) cerca unit root")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.7 -- VAR(2) k=3 con acople cruzado en lag 2
    A_list = [[[0.4,0.1,0.0],[0.1,0.4,0.1],[0.0,0.1,0.4]],[[0.05,0.1,0.0],[0.1,0.05,0.1],[0.0,0.1,0.05]]]
    Sigma  = [[1.0,0.2,0.0],[0.2,1.0,0.2],[0.0,0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=2: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.7 -- VAR(2) k=3 con acople cruzado en lag 2", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.7",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-B.7 -- VAR(2) k=3 con acople cruzado en lag 2\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(2)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(2)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-B.7 -- VAR(2) k=3 con acople cruzado en lag 2")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.8 -- VAR(5) k=2 memoria muy larga
    A_list = [[[0.20,0.0],[0.0,0.20]],[[0.15,0.0],[0.0,0.15]],[[0.10,0.0],[0.0,0.10]],[[0.07,0.0],[0.0,0.07]],[[0.05,0.0],[0.0,0.05]]]
    Sigma  = [[1.0,0.2],[0.2,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=5: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.8 -- VAR(5) k=2 memoria muy larga", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.8",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-B.8 -- VAR(5) k=2 memoria muy larga\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(5)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(5)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.8 -- VAR(5) k=2 memoria muy larga")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-B.9 -- VAR(1) k=2 casi-explosivo (max eig 0.99)
    A_list = [[[0.985,0.01],[0.01,0.98]]]
    Sigma  = [[1.0,0.3],[0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-B.9 -- VAR(1) k=2 casi-explosivo (max eig 0.99)", dgp, {}, CHECKS_VAR_NEAR_UNIT_ROOT)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-B.9",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-B.9 -- VAR(1) k=2 casi-explosivo (max eig 0.99)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-B.9 -- VAR(1) k=2 casi-explosivo (max eig 0.99)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-B.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-C — Dimensionalidad creciente (5 experimentos)

$$\mathbf{y}_t=A_1\,\mathbf{y}_{t-1}+\boldsymbol{\varepsilon}_t,\qquad \mathbf{y}_t\in\mathbb{R}^{k},\ k=3,\dots,6$$

Maldicion de dimensionalidad para el VAR frente a la transferencia de Chronos. Clasico: VAR(1).

**Experimentos:**
- **M-C.1** — VAR(1) k=3 tridiagonal · clasico: VAR(1)
- **M-C.2** — VAR(1) k=4 tridiagonal · clasico: VAR(1)
- **M-C.3** — VAR(1) k=5 tridiagonal · clasico: VAR(1)
- **M-C.4** — VAR(1) k=5 matriz densa · clasico: VAR(1)
- **M-C.5** — VAR(1) k=6 tridiagonal · clasico: VAR(1)

In [ ]:
try:
    # M-C.1 -- VAR(1) k=3 tridiagonal
    A_list = [[[0.5, 0.1, 0.0], [0.1, 0.5, 0.1], [0.0, 0.1, 0.5]]]
    Sigma  = [[1.0, 0.2, 0.0], [0.2, 1.0, 0.2], [0.0, 0.2, 1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-C.1 -- VAR(1) k=3 tridiagonal", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-C.1")
    log("\n" + "="*60 + "\nM-C.1 -- VAR(1) k=3 tridiagonal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-C.1 -- VAR(1) k=3 tridiagonal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-C.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-C.2 -- VAR(1) k=4 tridiagonal
    A_list = [[[0.4, 0.1, 0.0, 0.0], [0.1, 0.4, 0.1, 0.0], [0.0, 0.1, 0.4, 0.1], [0.0, 0.0, 0.1, 0.4]]]
    Sigma  = [[1.0, 0.2, 0.0, 0.0], [0.2, 1.0, 0.2, 0.0], [0.0, 0.2, 1.0, 0.2], [0.0, 0.0, 0.2, 1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-C.2 -- VAR(1) k=4 tridiagonal", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-C.2")
    log("\n" + "="*60 + "\nM-C.2 -- VAR(1) k=4 tridiagonal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4"], title="M-C.2 -- VAR(1) k=4 tridiagonal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-C.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-C.3 -- VAR(1) k=5 tridiagonal
    A_list = [[[0.3, 0.05, 0.0, 0.0, 0.0], [0.05, 0.3, 0.05, 0.0, 0.0], [0.0, 0.05, 0.3, 0.05, 0.0], [0.0, 0.0, 0.05, 0.3, 0.05], [0.0, 0.0, 0.0, 0.05, 0.3]]]
    Sigma  = [[1.0, 0.1, 0.0, 0.0, 0.0], [0.1, 1.0, 0.1, 0.0, 0.0], [0.0, 0.1, 1.0, 0.1, 0.0], [0.0, 0.0, 0.1, 1.0, 0.1], [0.0, 0.0, 0.0, 0.1, 1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-C.3 -- VAR(1) k=5 tridiagonal", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-C.3")
    log("\n" + "="*60 + "\nM-C.3 -- VAR(1) k=5 tridiagonal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4", "Y5"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4", "Y5"], title="M-C.3 -- VAR(1) k=5 tridiagonal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-C.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-C.4 -- VAR(1) k=5 matriz densa
    A_list = [[[0.3, 0.05, 0.05, 0.05, 0.05], [0.05, 0.3, 0.05, 0.05, 0.05], [0.05, 0.05, 0.3, 0.05, 0.05], [0.05, 0.05, 0.05, 0.3, 0.05], [0.05, 0.05, 0.05, 0.05, 0.3]]]
    Sigma  = [[1.0, 0.1, 0.0, 0.0, 0.0], [0.1, 1.0, 0.1, 0.0, 0.0], [0.0, 0.1, 1.0, 0.1, 0.0], [0.0, 0.0, 0.1, 1.0, 0.1], [0.0, 0.0, 0.0, 0.1, 1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-C.4 -- VAR(1) k=5 matriz densa", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-C.4", T_list=[100, 200])
    log("\n" + "="*60 + "\nM-C.4 -- VAR(1) k=5 matriz densa\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4", "Y5"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4", "Y5"], title="M-C.4 -- VAR(1) k=5 matriz densa")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-C.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-C.5 -- VAR(1) k=6 tridiagonal
    A_list = [[[0.3, 0.05, 0.0, 0.0, 0.0, 0.0], [0.05, 0.3, 0.05, 0.0, 0.0, 0.0], [0.0, 0.05, 0.3, 0.05, 0.0, 0.0], [0.0, 0.0, 0.05, 0.3, 0.05, 0.0], [0.0, 0.0, 0.0, 0.05, 0.3, 0.05], [0.0, 0.0, 0.0, 0.0, 0.05, 0.3]]]
    Sigma  = [[1.0, 0.1, 0.0, 0.0, 0.0, 0.0], [0.1, 1.0, 0.1, 0.0, 0.0, 0.0], [0.0, 0.1, 1.0, 0.1, 0.0, 0.0], [0.0, 0.0, 0.1, 1.0, 0.1, 0.0], [0.0, 0.0, 0.0, 0.1, 1.0, 0.1], [0.0, 0.0, 0.0, 0.0, 0.1, 1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-C.5 -- VAR(1) k=6 tridiagonal", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-C.5", T_list=[100, 200])
    log("\n" + "="*60 + "\nM-C.5 -- VAR(1) k=6 tridiagonal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4", "Y5", "Y6"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4", "Y5", "Y6"], title="M-C.5 -- VAR(1) k=6 tridiagonal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-C.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-D — VAR + GARCH diagonal (9 experimentos)

$$\mathbf{y}_t=A_1\,\mathbf{y}_{t-1}+\boldsymbol{\varepsilon}_t,\quad \varepsilon_{i,t}=\sigma_{i,t}z_{i,t},\quad \sigma_{i,t}^2=\omega_i+\alpha_i\varepsilon_{i,t-1}^2+\beta_i\sigma_{i,t-1}^2$$

VAR(1) con varianza condicional GARCH(1,1) diagonal por componente. Clasico: VAR + GARCH diagonal.

**Experimentos:**
- **M-D.1** — VAR(1) + GARCH baseline · clasico: VAR(1)+GARCH-diag
- **M-D.2** — GARCH reactivo · clasico: VAR(1)+GARCH-diag
- **M-D.3** — Casi IGARCH · clasico: VAR(1)+GARCH-diag
- **M-D.4** — Mean persistente + GARCH estandar · clasico: VAR(1)+GARCH-diag
- **M-D.5** — GARCH asimetrico entre ecuaciones · clasico: VAR(1)+GARCH-diag
- **M-D.6** — VAR+GARCH k=3 tridiagonal · clasico: VAR(1)+GARCH-diag
- **M-D.7** — GARCH reactivo bajo (alpha+beta=0.80) · clasico: VAR+GARCH
- **M-D.8** — Casi-IGARCH genuino (alpha+beta=0.99) · clasico: VAR+GARCH
- **M-D.9** — VAR alta interdep + GARCH estandar · clasico: VAR+GARCH

In [ ]:
try:
    # M-D.1 -- VAR(1) + GARCH baseline
    A1 = [[0.5,0.1],[0.1,0.5]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.1, 0.1],
                              alphas=[0.1, 0.15], betas=[0.8, 0.75])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.1 -- VAR(1) + GARCH baseline", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.1")
    log("\n" + "="*60 + "\nM-D.1 -- VAR(1) + GARCH baseline\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.1 -- VAR(1) + GARCH baseline")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.2 -- GARCH reactivo
    A1 = [[0.5,0.1],[0.1,0.5]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.1, 0.1],
                              alphas=[0.3, 0.3], betas=[0.6, 0.6])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.2 -- GARCH reactivo", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.2")
    log("\n" + "="*60 + "\nM-D.2 -- GARCH reactivo\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.2 -- GARCH reactivo")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.3 -- Casi IGARCH
    A1 = [[0.5,0.1],[0.1,0.5]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.05, 0.05],
                              alphas=[0.05, 0.05], betas=[0.9, 0.9])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.3 -- Casi IGARCH", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.3")
    log("\n" + "="*60 + "\nM-D.3 -- Casi IGARCH\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.3 -- Casi IGARCH")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.4 -- Mean persistente + GARCH estandar
    A1 = [[0.7,0.05],[0.05,0.3]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.1, 0.1],
                              alphas=[0.1, 0.1], betas=[0.8, 0.8])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.4 -- Mean persistente + GARCH estandar", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.4")
    log("\n" + "="*60 + "\nM-D.4 -- Mean persistente + GARCH estandar\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.4 -- Mean persistente + GARCH estandar")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.5 -- GARCH asimetrico entre ecuaciones
    A1 = [[0.3,0.2],[0.2,0.3]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.1, 0.1],
                              alphas=[0.2, 0.05], betas=[0.5, 0.9])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.5 -- GARCH asimetrico entre ecuaciones", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.5")
    log("\n" + "="*60 + "\nM-D.5 -- GARCH asimetrico entre ecuaciones\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.5 -- GARCH asimetrico entre ecuaciones")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.6 -- VAR+GARCH k=3 tridiagonal
    A1 = [[0.4, 0.1, 0.0], [0.1, 0.4, 0.1], [0.0, 0.1, 0.4]]
    dgp = VARGARCHDiagonalDGP(seed=SEED, A1=A1, omegas=[0.1, 0.1, 0.1],
                              alphas=[0.1, 0.1, 0.1], betas=[0.8, 0.8, 0.8])
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.6 -- VAR+GARCH k=3 tridiagonal", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.6")
    log("\n" + "="*60 + "\nM-D.6 -- VAR+GARCH k=3 tridiagonal\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)+GARCH-diag", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)+GARCH-diag")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-D.6 -- VAR+GARCH k=3 tridiagonal")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.7 -- GARCH reactivo bajo (alpha+beta=0.80)
    A1 = [[0.5,0.1],[0.1,0.5]]
    dgp = VARGARCHDiagonalDGP(
        seed=SEED, A1=A1,
        omegas=[0.2, 0.2], alphas=[0.2, 0.2], betas=[0.6, 0.6],
    )
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.7 -- GARCH reactivo bajo (alpha+beta=0.80)", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.7")
    log("\n" + "="*60 + "\nM-D.7 -- GARCH reactivo bajo (alpha+beta=0.80)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR+GARCH", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR+GARCH")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.7 -- GARCH reactivo bajo (alpha+beta=0.80)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.8 -- Casi-IGARCH genuino (alpha+beta=0.99)
    A1 = [[0.5,0.1],[0.1,0.5]]
    dgp = VARGARCHDiagonalDGP(
        seed=SEED, A1=A1,
        omegas=[0.01, 0.01], alphas=[0.04, 0.04], betas=[0.95, 0.95],
    )
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.8 -- Casi-IGARCH genuino (alpha+beta=0.99)", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.8")
    log("\n" + "="*60 + "\nM-D.8 -- Casi-IGARCH genuino (alpha+beta=0.99)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR+GARCH", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR+GARCH")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.8 -- Casi-IGARCH genuino (alpha+beta=0.99)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-D.9 -- VAR alta interdep + GARCH estandar
    A1 = [[0.4,0.4],[0.4,0.4]]
    dgp = VARGARCHDiagonalDGP(
        seed=SEED, A1=A1,
        omegas=[0.1, 0.1], alphas=[0.1, 0.1], betas=[0.8, 0.8],
    )
    make_cl = lambda T: VARGARCHDiagonalModel(seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-D.9 -- VAR alta interdep + GARCH estandar", dgp, {}, CHECKS_VAR_GARCH)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-D.9")
    log("\n" + "="*60 + "\nM-D.9 -- VAR alta interdep + GARCH estandar\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR+GARCH", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR+GARCH")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-D.9 -- VAR alta interdep + GARCH estandar")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-D.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-E — Cointegracion VECM bivariado (8 experimentos)

$$\Delta\mathbf{y}_t=\alpha\,\boldsymbol{\beta}'\,\mathbf{y}_{t-1}+\Gamma_1\,\Delta\mathbf{y}_{t-1}+\boldsymbol{\varepsilon}_t$$

Rango de cointegracion 1, k = 2. Se varian alpha (velocidad de ajuste), beta (vector de cointegracion), Gamma_1 y Sigma. Clasico: VECM(r=1).

**Experimentos:**
- **M-E.1** — VECM baseline ajuste medio · clasico: VECM(r=1)
- **M-E.2** — VECM ajuste lento · clasico: VECM(r=1)
- **M-E.3** — VECM ajuste rapido · clasico: VECM(r=1)
- **M-E.4** — VECM cointegracion no-1:1 · clasico: VECM(r=1)
- **M-E.5** — VECM dinamica corta cruzada + Sigma corr · clasico: VECM(r=1)
- **M-E.6** — VECM sin dinamica corta (Gamma1=0) · clasico: VECM
- **M-E.7** — VECM ajuste asimetrico (Y2 no ajusta) · clasico: VECM
- **M-E.8** — VECM con Sigma correlacionada negativa · clasico: VECM

In [ ]:
try:
    # M-E.1 -- VECM baseline ajuste medio
    dgp = VECMBivariateDGP(seed=SEED, alpha=[-0.4, 0.2], beta=[1.0, -1.0],
                            Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, 0.0], [0.0, 1.0]])
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.1 -- VECM baseline ajuste medio", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.1")
    log("\n" + "="*60 + "\nM-E.1 -- VECM baseline ajuste medio\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM(r=1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM(r=1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.1 -- VECM baseline ajuste medio")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.2 -- VECM ajuste lento
    dgp = VECMBivariateDGP(seed=SEED, alpha=[-0.1, 0.05], beta=[1.0, -1.0],
                            Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, 0.0], [0.0, 1.0]])
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.2 -- VECM ajuste lento", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.2")
    log("\n" + "="*60 + "\nM-E.2 -- VECM ajuste lento\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM(r=1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM(r=1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.2 -- VECM ajuste lento")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.3 -- VECM ajuste rapido
    dgp = VECMBivariateDGP(seed=SEED, alpha=[-0.7, 0.3], beta=[1.0, -1.0],
                            Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, 0.0], [0.0, 1.0]])
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.3 -- VECM ajuste rapido", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.3")
    log("\n" + "="*60 + "\nM-E.3 -- VECM ajuste rapido\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM(r=1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM(r=1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.3 -- VECM ajuste rapido")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.4 -- VECM cointegracion no-1:1
    dgp = VECMBivariateDGP(seed=SEED, alpha=[-0.4, 0.2], beta=[1.0, -2.0],
                            Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, 0.0], [0.0, 1.0]])
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.4 -- VECM cointegracion no-1:1", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.4")
    log("\n" + "="*60 + "\nM-E.4 -- VECM cointegracion no-1:1\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM(r=1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM(r=1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.4 -- VECM cointegracion no-1:1")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.5 -- VECM dinamica corta cruzada + Sigma corr
    dgp = VECMBivariateDGP(seed=SEED, alpha=[-0.4, 0.2], beta=[1.0, -1.0],
                            Gamma1=[[0.5, 0.2], [0.2, 0.5]], Sigma=[[1.0, 0.5], [0.5, 1.0]])
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.5 -- VECM dinamica corta cruzada + Sigma corr", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.5")
    log("\n" + "="*60 + "\nM-E.5 -- VECM dinamica corta cruzada + Sigma corr\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM(r=1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM(r=1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.5 -- VECM dinamica corta cruzada + Sigma corr")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.6 -- VECM sin dinamica corta (Gamma1=0)
    dgp = VECMBivariateDGP(
        seed=SEED,
        alpha=[-0.4, 0.2], beta=[1.0, -1.0],
        Gamma1=[[0.0, 0.0], [0.0, 0.0]], Sigma=[[1.0, 0.0], [0.0, 1.0]],
    )
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.6 -- VECM sin dinamica corta (Gamma1=0)", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.6")
    log("\n" + "="*60 + "\nM-E.6 -- VECM sin dinamica corta (Gamma1=0)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.6 -- VECM sin dinamica corta (Gamma1=0)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.7 -- VECM ajuste asimetrico (Y2 no ajusta)
    dgp = VECMBivariateDGP(
        seed=SEED,
        alpha=[-0.5, 0.0], beta=[1.0, -1.0],
        Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, 0.0], [0.0, 1.0]],
    )
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.7 -- VECM ajuste asimetrico (Y2 no ajusta)", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.7")
    log("\n" + "="*60 + "\nM-E.7 -- VECM ajuste asimetrico (Y2 no ajusta)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.7 -- VECM ajuste asimetrico (Y2 no ajusta)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-E.8 -- VECM con Sigma correlacionada negativa
    dgp = VECMBivariateDGP(
        seed=SEED,
        alpha=[-0.4, 0.2], beta=[1.0, -1.0],
        Gamma1=[[0.3, 0.0], [0.0, 0.3]], Sigma=[[1.0, -0.5], [-0.5, 1.0]],
    )
    make_cl = lambda T: VECMModel(coint_rank=1, k_ar_diff=1, n_sim=200, seed=SEED)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-E.8 -- VECM con Sigma correlacionada negativa", dgp, {}, CHECKS_VECM)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-E.8")
    log("\n" + "="*60 + "\nM-E.8 -- VECM con Sigma correlacionada negativa\n" + "="*60)
    build_grid_table_mv(res, classical_name="VECM", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VECM")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-E.8 -- VECM con Sigma correlacionada negativa")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-E.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-F — VAR con eigenvalores complejos (ciclos endogenos) (3 experimentos)

$$\mathbf{y}_t=A_1\,\mathbf{y}_{t-1}+\boldsymbol{\varepsilon}_t,\qquad A_1\ \text{con autovalores complejos conjugados}$$

A_1 induce ciclos periodicos endogenos (sustituto del estacional vectorial). Clasico: VAR(1).

**Experimentos:**
- **M-F.1** — Ciclo lento · clasico: VAR(1)
- **M-F.2** — Ciclo marcado · clasico: VAR(1)
- **M-F.3** — Flip periodo-2 · clasico: VAR(1)

In [ ]:
try:
    # M-F.1 -- Ciclo lento
    A_list = [[[0.7,-0.5],[0.5,0.7]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-F.1 -- Ciclo lento", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-F.1")
    log("\n" + "="*60 + "\nM-F.1 -- Ciclo lento\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-F.1 -- Ciclo lento")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-F.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-F.2 -- Ciclo marcado
    A_list = [[[0.5,-0.8],[0.8,0.5]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-F.2 -- Ciclo marcado", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-F.2")
    log("\n" + "="*60 + "\nM-F.2 -- Ciclo marcado\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-F.2 -- Ciclo marcado")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-F.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-F.3 -- Flip periodo-2
    A_list = [[[0.0,0.9],[0.9,0.0]]]
    Sigma  = [[1.0,0.0],[0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-F.3 -- Flip periodo-2", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-F.3")
    log("\n" + "="*60 + "\nM-F.3 -- Flip periodo-2\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2"], title="M-F.3 -- Flip periodo-2")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-F.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

## Bloque M-G — Patrones de acople sistematicos (9 experimentos)

$$\mathbf{y}_t=\sum_{i=1}^{p}A_i\,\mathbf{y}_{t-i}+\boldsymbol{\varepsilon}_t$$

Estructuras de acople diversas: block-diagonal, cadena triangular, hub-and-spoke, denso uniforme, retroalimentacion negativa, dim x lag, alta dimensionalidad y eigenvalores mixtos. Clasico: VAR(p).

**Experimentos:**
- **M-G.1** — Block-diagonal k=4 (dos subsistemas independientes) · clasico: VAR(1)
- **M-G.2** — Cadena triangular Y1->Y2->Y3 · clasico: VAR(1)
- **M-G.3** — Acople denso uniforme k=3 · clasico: VAR(1)
- **M-G.4** — Hub-and-spoke k=4 (Y1 influye a Y2..Y4) · clasico: VAR(1)
- **M-G.5** — Retroalimentacion negativa densa k=3 · clasico: VAR(1)
- **M-G.6** — VAR(2) k=4 sparse · clasico: VAR(2)
- **M-G.7** — VAR(1) k=8 banded (alta dimensionalidad real) · clasico: VAR(1)
- **M-G.8** — Eigenvalores mixtos reales+complejos k=3 · clasico: VAR(1)
- **M-G.9** — Ciclo endogeno k=3 · clasico: VAR(1)

In [ ]:
try:
    # M-G.1 -- Block-diagonal k=4 (dos subsistemas independientes)
    A_list = [[[0.5,0.1,0.0,0.0],[0.1,0.5,0.0,0.0],[0.0,0.0,0.4,0.2],[0.0,0.0,0.2,0.4]]]
    Sigma  = [[1.0,0.0,0.0,0.0],[0.0,1.0,0.0,0.0],[0.0,0.0,1.0,0.0],[0.0,0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.1 -- Block-diagonal k=4 (dos subsistemas independientes)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.1",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-G.1 -- Block-diagonal k=4 (dos subsistemas independientes)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4"], title="M-G.1 -- Block-diagonal k=4 (dos subsistemas independientes)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.2 -- Cadena triangular Y1->Y2->Y3
    A_list = [[[0.5,0.0,0.0],[0.3,0.4,0.0],[0.0,0.3,0.4]]]
    Sigma  = [[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.2 -- Cadena triangular Y1->Y2->Y3", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.2")
    log("\n" + "="*60 + "\nM-G.2 -- Cadena triangular Y1->Y2->Y3\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-G.2 -- Cadena triangular Y1->Y2->Y3")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.3 -- Acople denso uniforme k=3
    A_list = [[[0.25,0.25,0.25],[0.25,0.25,0.25],[0.25,0.25,0.25]]]
    Sigma  = [[1.0,0.3,0.3],[0.3,1.0,0.3],[0.3,0.3,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.3 -- Acople denso uniforme k=3", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.3",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-G.3 -- Acople denso uniforme k=3\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-G.3 -- Acople denso uniforme k=3")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.4 -- Hub-and-spoke k=4 (Y1 influye a Y2..Y4)
    A_list = [[[0.5,0.0,0.0,0.0],[0.3,0.4,0.0,0.0],[0.3,0.0,0.4,0.0],[0.3,0.0,0.0,0.4]]]
    Sigma  = [[1.0,0.0,0.0,0.0],[0.0,1.0,0.0,0.0],[0.0,0.0,1.0,0.0],[0.0,0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.4 -- Hub-and-spoke k=4 (Y1 influye a Y2..Y4)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.4",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-G.4 -- Hub-and-spoke k=4 (Y1 influye a Y2..Y4)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4"], title="M-G.4 -- Hub-and-spoke k=4 (Y1 influye a Y2..Y4)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.5 -- Retroalimentacion negativa densa k=3
    A_list = [[[0.5,-0.15,-0.15],[-0.15,0.5,-0.15],[-0.15,-0.15,0.5]]]
    Sigma  = [[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.5 -- Retroalimentacion negativa densa k=3", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.5")
    log("\n" + "="*60 + "\nM-G.5 -- Retroalimentacion negativa densa k=3\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-G.5 -- Retroalimentacion negativa densa k=3")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.6 -- VAR(2) k=4 sparse
    A_list = [[[0.4,0.1,0.0,0.0],[0.1,0.4,0.1,0.0],[0.0,0.1,0.4,0.1],[0.0,0.0,0.1,0.4]],[[0.1,0.0,0.0,0.0],[0.0,0.1,0.0,0.0],[0.0,0.0,0.1,0.0],[0.0,0.0,0.0,0.1]]]
    Sigma  = [[1.0,0.0,0.0,0.0],[0.0,1.0,0.0,0.0],[0.0,0.0,1.0,0.0],[0.0,0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=2: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.6 -- VAR(2) k=4 sparse", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.6",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-G.6 -- VAR(2) k=4 sparse\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(2)", var_names=["Y1", "Y2", "Y3", "Y4"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(2)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4"], title="M-G.6 -- VAR(2) k=4 sparse")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.7 -- VAR(1) k=8 banded (alta dimensionalidad real)
    A_list = [[[0.4, 0.05, 0.02, 0.0, 0.0, 0.0, 0.0, 0.0], [0.05, 0.4, 0.05, 0.02, 0.0, 0.0, 0.0, 0.0], [0.02, 0.05, 0.4, 0.05, 0.02, 0.0, 0.0, 0.0], [0.0, 0.02, 0.05, 0.4, 0.05, 0.02, 0.0, 0.0], [0.0, 0.0, 0.02, 0.05, 0.4, 0.05, 0.02, 0.0], [0.0, 0.0, 0.0, 0.02, 0.05, 0.4, 0.05, 0.02], [0.0, 0.0, 0.0, 0.0, 0.02, 0.05, 0.4, 0.05], [0.0, 0.0, 0.0, 0.0, 0.0, 0.02, 0.05, 0.4]]]
    Sigma  = [[1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.7 -- VAR(1) k=8 banded (alta dimensionalidad real)", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.7",
                     T_list=[200])
    log("\n" + "="*60 + "\nM-G.7 -- VAR(1) k=8 banded (alta dimensionalidad real)\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3", "Y4", "Y5", "Y6", "Y7", "Y8"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3", "Y4", "Y5", "Y6", "Y7", "Y8"], title="M-G.7 -- VAR(1) k=8 banded (alta dimensionalidad real)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.8 -- Eigenvalores mixtos reales+complejos k=3
    A_list = [[[0.5,-0.6,0.0],[0.6,0.5,0.0],[0.0,0.0,0.7]]]
    Sigma  = [[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.8 -- Eigenvalores mixtos reales+complejos k=3", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.8")
    log("\n" + "="*60 + "\nM-G.8 -- Eigenvalores mixtos reales+complejos k=3\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-G.8 -- Eigenvalores mixtos reales+complejos k=3")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # M-G.9 -- Ciclo endogeno k=3
    A_list = [[[0.7,-0.5,0.0],[0.5,0.7,0.1],[0.0,0.1,0.5]]]
    Sigma  = [[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]]
    dgp = VARDGP(seed=SEED, A_list=A_list, Sigma=Sigma)
    make_cl = lambda T, lags=1: VARModel(lags=lags)
    make_models = lambda T: [make_cl(T), chronos_mv]
    verify_dgp_mv("M-G.9 -- Ciclo endogeno k=3", dgp, {}, CHECKS_VAR)
    res = run_exp_mv(dgp, make_models, {}, exp_id="M-G.9",
                     T_list=[100, 200])
    log("\n" + "="*60 + "\nM-G.9 -- Ciclo endogeno k=3\n" + "="*60)
    build_grid_table_mv(res, classical_name="VAR(1)", var_names=["Y1", "Y2", "Y3"])
    print("\n--- Tabla 2: metricas multivariadas conjuntas (Trace MSFE + avgCRPS) ---")
    build_grid_table_mv_joint(res, classical_name="VAR(1)")
    plot_simulation_mv(dgp, [make_cl(200), chronos_mv], {},
                       var_names=["Y1", "Y2", "Y3"], title="M-G.9 -- Ciclo endogeno k=3")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA M-G.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
# Recorre todos los CSV cacheados y arma summary_table_all
import re

summary_rows = []
csv_files = sorted(RESULTS.glob("exp_*.csv"))
log(f"Encontrados {len(csv_files)} CSV en {RESULTS}")

pattern = re.compile(r"exp_(?P<exp>.+?)_T(?P<T>\d+)_R(?P<R>\d+)\.csv")

for csv_path in csv_files:
    m = pattern.match(csv_path.name)
    if not m:
        continue
    exp_id = m.group("exp").replace("_", ".", 1)  # M-A_1 -> M-A.1
    T = int(m.group("T"))
    R = int(m.group("R"))
    bloque = exp_id.split(".")[0]

    res = _load_results_mv(csv_path)
    blk_data = compute_blocks_mv(res)
    for mname, var_blks in blk_data.items():
        for var_idx, blks in var_blks.items():
            for bname, s in blks.items():
                if var_idx >= 0:
                    # Fila per-variable
                    vname = f"Y{var_idx+1}"
                    row = {
                        "Bloque": bloque, "Exp": exp_id, "T": T, "R": R,
                        "Modelo": mname, "Variable": vname, "h-block": bname,
                        "rmse":     float(s["rmse"])     if "rmse"     in s.index and pd.notna(s["rmse"])     else np.nan,
                        "bias":     float(s["bias"])     if "bias"     in s.index and pd.notna(s["bias"])     else np.nan,
                        "variance": float(s["variance"]) if "variance" in s.index and pd.notna(s["variance"]) else np.nan,
                        "crps":     float(s["crps"])     if "crps"     in s.index and pd.notna(s["crps"])     else np.nan,
                        "trace_msfe": np.nan,
                        "avg_crps":   np.nan,
                    }
                else:
                    # Fila joint (var=-1)
                    row = {
                        "Bloque": bloque, "Exp": exp_id, "T": T, "R": R,
                        "Modelo": mname, "Variable": "JOINT", "h-block": bname,
                        "rmse": np.nan, "bias": np.nan, "variance": np.nan, "crps": np.nan,
                        "trace_msfe": float(s["trace_msfe"]) if "trace_msfe" in s.index and pd.notna(s["trace_msfe"]) else np.nan,
                        "avg_crps":   float(s["avg_crps"])   if "avg_crps"   in s.index and pd.notna(s["avg_crps"])   else np.nan,
                    }
                summary_rows.append(row)

summary_table_all = pd.DataFrame(summary_rows)
log(f"summary_table_all: {summary_table_all.shape}")
display(summary_table_all.head(20))

# --- Tabla 1: RMSE per-variable promedio por (Bloque, Modelo, T) ---
per_var = summary_table_all[summary_table_all["Variable"] != "JOINT"]
if len(per_var) > 0:
    agg = (per_var
           .groupby(["Bloque", "Modelo", "T"], as_index=False)["rmse"]
           .mean()
           .pivot(index=["Bloque", "Modelo"], columns="T", values="rmse"))
    log("\n--- RMSE per-variable promedio por (Bloque, Modelo, T) ---")
    display(agg.style.format(precision=4, na_rep="—")
             .background_gradient(cmap="YlOrRd", axis=1))

# --- Tabla 2: Trace MSFE y avgCRPS joint promedio por (Bloque, Modelo, T) ---
joint = summary_table_all[summary_table_all["Variable"] == "JOINT"]
if len(joint) > 0:
    for metric in ["trace_msfe", "avg_crps"]:
        agg = (joint
               .groupby(["Bloque", "Modelo", "T"], as_index=False)[metric]
               .mean()
               .pivot(index=["Bloque", "Modelo"], columns="T", values=metric))
        log(f"\n--- {metric.upper()} joint promedio por (Bloque, Modelo, T) ---")
        display(agg.style.format(precision=4, na_rep="—")
                 .background_gradient(cmap="YlOrRd", axis=1))
else:
    log("\n[!] No se encontraron filas joint en el summary. "
        "Los CSV pueden haber sido generados con una version anterior del engine; "
        "borrar y re-ejecutar para incluir las metricas multivariadas conjuntas.")